In [2]:
import json

from shared.utils.backup import Backup
from smyrooms.client.distributor_client import DistributorRulesClient
from smyrooms.client.distributor_client import RuleFilter

backup = Backup.for_project("smyrooms")

rules_client = DistributorRulesClient(
    organization_id="lgt",
    product_id="hot",
    environment="pro",
)

In [3]:
backup_dir = backup.paths.get_data_dir("backups")

with open(backup_dir / "2025-09-22T09-52-26_level_closes.json", encoding="utf-8") as f:
    previous_week = json.load(f)


with open(backup_dir / "2025-09-29T11-27-10_level_closes.json", encoding="utf-8") as f:
    current_week = json.load(f)


print(len(previous_week), len(current_week))

4351 4294


In [ ]:
excl = [
    "2762", "499300", "394337", "435120", "394689", "394917", "493709", "395623",
    "492821", "394173", "833022", "394510", "394745", "670519", "528277", "803961",
    "10338", "529117", "394287", "464455", "394691", "432422", "445980", "303659",
    "683960", "707185", "395622", "427640", "510539", "394896", "395733", "482048",
    "707184", "518565", "655513", "509914", "482337", "802100", "395442", "693597",
    "704757", "211282", "463397", "549364", "736961", "394656", "395674", "394762",
    "395221", "707182", "395441", "526370", "477105", "706134", "697099", "936846",
    "946175", "386959", "931562", "906925", "914796", "2694", "475809", "840285",
    "707186", "395626", "938", "658677", "394679", "745453", "845988", "291912",
    "16629", "939494", "913937", "745425", "807476", "395615", "499819", "910968","395530",
    "736025", "926227", "442638", "432611", "395618", "491415", "468559", "343302", "395630",
    "468383", "2335", "894614", "805340", "395029", "395620", "707181", "527208", "395708",
    "395617", "299372", "395612", "727264", "726670", "811340", "395621", "395265", "520833",
    "395088", "707180", "745092", "395718", "394751", "872853", "394505", "693596", "395558",
    "754628", "634371", "728722", "791260", "482642", "607966", "863332", "394569", "791416",
    "99061", "394464", "621607", "395613", "430830", "427634", "684889", "395334", "823885",
    "510163", "467283", "394908", "806583", "884750", "932386", "522450", "460292", "395619",
    "394931", "311391", "537885", "184398", "1003940", "395324", "693502", "684792", "395628",
    "894597", "856492", "792387", "930761", "597353", "395629", "811867", "395616", "395134",
    "707183", "395129", "395804", "684890", "395163", "500589", "468591", "106981", "850670",
    "395803", "636354", "30462", "467240", "863169", "727262", "691275", "2636"
]


In [11]:
previous_week = (
    RuleFilter(previous_week).apply_analysis_defaults().exclude_field_in("id", excl).to_dict()
)
current_week = (
    RuleFilter(current_week).apply_analysis_defaults().exclude_field_in("id", excl).to_dict()
)

print(f"Previous week: {len(previous_week)} rules")
print(f"Current week: {len(current_week)} rules")

Previous week: 1571 rules
Current week: 1585 rules


In [12]:
# Perform the actual comparison between previous and current week
changes = {"added": [], "modified": [], "removed": []}

# Find new and modified rules
for rule_id, current_rule in current_week.items():
    if rule_id not in previous_week:
        changes["added"].append({"rule_id": rule_id, "rule": current_rule})
    elif current_rule != previous_week[rule_id]:
        changes["modified"].append(
            {
                "rule_id": rule_id,
                "previous": previous_week[rule_id],
                "current": current_rule,
            }
        )

# Find removed rules
for rule_id, prev_rule in previous_week.items():
    if rule_id not in current_week:
        changes["removed"].append({"rule_id": rule_id, "rule": prev_rule})

print("Comparison results:")
print(f"  Added rules: {len(changes['added'])}")
print(f"  Modified rules: {len(changes['modified'])}")
print(f"  Removed rules: {len(changes['removed'])}")

Comparison results:
  Added rules: 29
  Modified rules: 21
  Removed rules: 15


In [14]:
removed_rules = set(previous_week.keys()) - set(current_week.keys())

In [15]:
len(removed_rules)

15

In [6]:
def analyze_rule_changes(previous_rule: dict, current_rule: dict) -> dict:
    """Analyze specific changes between two rule versions.

    Args:
        previous_rule: The previous version of the rule
        current_rule: The current version of the rule

    Returns:
        Dictionary containing detailed change information
    """
    changes = {
        "rule_id": current_rule.get("id"),
        "field_changes": [],
        "summary": {"changed_fields": 0, "change_types": []},
    }

    def compare_nested_dict(prev_dict, curr_dict, path=""):
        """Recursively compare nested dictionaries."""
        all_fields = set(prev_dict.keys()) | set(curr_dict.keys())

        for field in all_fields:
            current_path = f"{path}.{field}" if path else field

            if field not in prev_dict:
                # Field was added
                changes["field_changes"].append(
                    {
                        "field": current_path,
                        "change_type": "added",
                        "previous_value": None,
                        "current_value": curr_dict[field],
                    }
                )
                changes["summary"]["change_types"].append("added")

            elif field not in curr_dict:
                # Field was removed
                changes["field_changes"].append(
                    {
                        "field": current_path,
                        "change_type": "removed",
                        "previous_value": prev_dict[field],
                        "current_value": None,
                    }
                )
                changes["summary"]["change_types"].append("removed")

            elif isinstance(prev_dict[field], dict) and isinstance(
                curr_dict[field], dict
            ):
                # Both are dicts, recurse
                compare_nested_dict(prev_dict[field], curr_dict[field], current_path)

            elif isinstance(prev_dict[field], list) and isinstance(
                curr_dict[field], list
            ):
                # Compare lists
                if prev_dict[field] != curr_dict[field]:
                    prev_set = (
                        set(prev_dict[field])
                        if all(
                            isinstance(x, (str, int, float)) for x in prev_dict[field]
                        )
                        else prev_dict[field]
                    )
                    curr_set = (
                        set(curr_dict[field])
                        if all(
                            isinstance(x, (str, int, float)) for x in curr_dict[field]
                        )
                        else curr_dict[field]
                    )

                    if isinstance(prev_set, set) and isinstance(curr_set, set):
                        added_items = curr_set - prev_set
                        removed_items = prev_set - curr_set

                        if added_items or removed_items:
                            changes["field_changes"].append(
                                {
                                    "field": current_path,
                                    "change_type": "list_modified",
                                    "previous_value": list(prev_set),
                                    "current_value": list(curr_set),
                                    "added_items": list(added_items),
                                    "removed_items": list(removed_items),
                                }
                            )
                            changes["summary"]["change_types"].append("list_modified")
                    else:
                        # Complex list comparison
                        changes["field_changes"].append(
                            {
                                "field": current_path,
                                "change_type": "list_changed",
                                "previous_value": prev_dict[field],
                                "current_value": curr_dict[field],
                            }
                        )
                        changes["summary"]["change_types"].append("list_changed")

            elif prev_dict[field] != curr_dict[field]:
                # Simple value change
                changes["field_changes"].append(
                    {
                        "field": current_path,
                        "change_type": "modified",
                        "previous_value": prev_dict[field],
                        "current_value": curr_dict[field],
                    }
                )
                changes["summary"]["change_types"].append("modified")

    # Start the comparison
    compare_nested_dict(previous_rule, current_rule)

    # Update summary
    changes["summary"]["changed_fields"] = len(changes["field_changes"])
    changes["summary"]["change_types"] = list(set(changes["summary"]["change_types"]))

    return changes

In [7]:
# Enhanced analysis of modified rules
detailed_changes = []

for modified_rule in changes["modified"]:
    rule_id = modified_rule["rule_id"]
    previous = modified_rule["previous"]
    current = modified_rule["current"]

    # Get detailed analysis
    detailed_change = analyze_rule_changes(previous, current)
    detailed_changes.append(detailed_change)

print(f"Found {len(detailed_changes)} modified rules with detailed analysis:")

# Display summary of change types
change_type_summary = {}
for detail in detailed_changes:
    for change_type in detail["summary"]["change_types"]:
        change_type_summary[change_type] = change_type_summary.get(change_type, 0) + 1

print("Change type summary:")
for change_type, count in change_type_summary.items():
    print(f"  {change_type}: {count} rules")

Found 21 modified rules with detailed analysis:
Change type summary:
  modified: 21 rules
  list_modified: 18 rules


In [8]:
# Example: Analyze specific rule changes in detail
if detailed_changes:
    # Look at the first modified rule as an example
    sample_change = detailed_changes[0]

    print(f"\nDetailed analysis for rule {sample_change['rule_id']}:")
    print(f"Number of field changes: {sample_change['summary']['changed_fields']}")
    print(f"Change types: {sample_change['summary']['change_types']}")

    print("\nField-by-field changes:")
    for field_change in sample_change['field_changes']:
        print(f"  Field: {field_change['field']}")
        print(f"  Change type: {field_change['change_type']}")
        print(f"  Previous: {field_change['previous_value']}")
        print(f"  Current: {field_change['current_value']}")

        if field_change['change_type'] == 'list_modified':
            if field_change.get('added_items'):
                print(f"  Added items: {field_change['added_items']}")
            if field_change.get('removed_items'):
                print(f"  Removed items: {field_change['removed_items']}")
        print("  ---")


Detailed analysis for rule 860871:
Number of field changes: 2
Change types: ['modified', 'list_modified']

Field-by-field changes:
  Field: lastDate
  Change type: modified
  Previous: 04-04-2025 13:49:00
  Current: 23-09-2025 11:17:05
  ---
  Field: lvl.hot.l
  Change type: list_modified
  Previous: ['573', '351', '204', '1368', '4616', '712', '4759', '4629', '537', '798', '153', '1521', '4792', '1014', '4362']
  Current: ['573', '351', '204', '712', '4759', '4629', '537', '798', '153', '1521', '4792', '1014', '4362']
  Removed items: ['4616', '1368']
  ---


In [9]:
# Save detailed analysis to backup for future reference
from datetime import datetime

detailed_analysis_report = {
    "analysis_date": datetime.now().isoformat(),
    "comparison_period": {
        "previous": "2025-09-22T09-52-26",
        "current": "2025-09-29T11-27-10"
    },
    "summary": {
        "total_modified_rules": len(detailed_changes),
        "change_type_breakdown": change_type_summary
    },
    "detailed_changes": detailed_changes
}

# Save the detailed analysis
analysis_file = backup.save_to_json(detailed_analysis_report, "level_closes_detailed_analysis")
print(f"\nDetailed analysis saved to: {analysis_file}")


Detailed analysis saved to: /home/ubuntu/notebooks/smyrooms/data/backups/2025-09-30T17-07-29_level_closes_detailed_analysis.json


In [16]:
# Simplified Rule Change Analysis - Fixed Version
def analyze_rule_changes_simple(previous_rules_dict, current_rules_dict):
    """Simple analysis focusing only on client access changes"""

    changes = []

    for rule_id in current_rules_dict:
        if rule_id not in previous_rules_dict:
            continue  # Skip new rules for now

        prev_rule = previous_rules_dict[rule_id]
        curr_rule = current_rules_dict[rule_id]

        # Focus only on client list changes (lvl.cli.l) - handle None values
        prev_client_list = prev_rule.get("lvl", {}).get("cli", {}).get("l", [])
        curr_client_list = curr_rule.get("lvl", {}).get("cli", {}).get("l", [])

        # Handle None values
        prev_clients = set(prev_client_list or [])
        curr_clients = set(curr_client_list or [])

        if prev_clients != curr_clients:
            opened_clients = list(prev_clients - curr_clients)  # Removed from restriction = opened
            closed_clients = list(curr_clients - prev_clients)  # Added to restriction = closed

            if opened_clients or closed_clients:
                change = {
                    "rule_id": rule_id,
                    "change_type": "clients_opened" if opened_clients else "clients_closed",
                    "clients_affected": {
                        "opened": opened_clients,
                        "closed": closed_clients,
                        "opened_count": len(opened_clients),
                        "closed_count": len(closed_clients),
                    },
                    "rule_scope": {
                        "hotels": curr_rule.get("lvl", {}).get("hot", {}),
                        "providers": curr_rule.get("lvl", {}).get("prv", {}),
                    },
                }
                changes.append(change)

    return changes


# Run the simplified analysis
simple_changes = analyze_rule_changes_simple(previous_week, current_week)

print(f"Found {len(simple_changes)} rules with client access changes")
print("\nSample changes:")
for i, change in enumerate(simple_changes[:5]):
    print(
        f"\nRule {change['rule_id']}: {change['clients_affected']['opened_count']} clients opened, {change['clients_affected']['closed_count']} clients closed"
    )
    if change["clients_affected"]["opened"]:
        print(
            f"  Opened clients: {change['clients_affected']['opened'][:3]}{'...' if len(change['clients_affected']['opened']) > 3 else ''}"
        )

# Summary statistics
openings = [c for c in simple_changes if c["clients_affected"]["opened_count"] > 0]
closings = [c for c in simple_changes if c["clients_affected"]["closed_count"] > 0]

print("\n--- Summary ---")
print(f"Rules with client openings: {len(openings)}")
print(f"Rules with client closings: {len(closings)}")
print(
    f"Total clients opened: {sum(c['clients_affected']['opened_count'] for c in openings)}"
)
print(
    f"Total clients closed: {sum(c['clients_affected']['closed_count'] for c in closings)}"
)

Found 5 rules with client access changes

Sample changes:

Rule 1008402: 0 clients opened, 5 clients closed

Rule 464445: 4 clients opened, 1 clients closed
  Opened clients: ['5226', '11945', '8550']...

Rule 395522: 3 clients opened, 1 clients closed
  Opened clients: ['11732', '11731', '3713']

Rule 395308: 6 clients opened, 3 clients closed
  Opened clients: ['5060', '11656', '11732']...

Rule 506996: 0 clients opened, 1 clients closed

--- Summary ---
Rules with client openings: 3
Rules with client closings: 5
Total clients opened: 13
Total clients closed: 11


In [18]:
# Clean Simplified Rule Change Analysis - Fixed Version
def analyze_rule_changes_clean(previous_rules_dict, current_rules_dict):
    """Simple analysis with clean output format"""

    changes = []

    for rule_id in current_rules_dict:
        if rule_id not in previous_rules_dict:
            continue  # Skip new rules for now

        prev_rule = previous_rules_dict[rule_id]
        curr_rule = current_rules_dict[rule_id]

        # Focus only on client list changes (lvl.cli.l) - handle None values
        prev_client_list = prev_rule.get("lvl", {}).get("cli", {}).get("l", [])
        curr_client_list = curr_rule.get("lvl", {}).get("cli", {}).get("l", [])

        # Handle None values
        prev_clients = set(prev_client_list or [])
        curr_clients = set(curr_client_list or [])

        if prev_clients != curr_clients:
            opened_clients = list(prev_clients - curr_clients)  # Removed from restriction = opened
            closed_clients = list(curr_clients - prev_clients)  # Added to restriction = closed

            # Handle opened clients
            if opened_clients:
                change = {
                    "rule_id": rule_id,
                    "change_type": "clients_opened",
                    "clients_affected": {
                        "opened": opened_clients,
                        "count": len(opened_clients)
                    },
                    "rule_scope": {
                        "hotels": {
                            "type": curr_rule.get("lvl", {}).get("hot", {}).get("t", 0),
                            "list": curr_rule.get("lvl", {}).get("hot", {}).get("l", []) or []
                        },
                        "providers": {
                            "type": curr_rule.get("lvl", {}).get("prv", {}).get("t", 0),
                            "list": curr_rule.get("lvl", {}).get("prv", {}).get("l", []) or []
                        }
                    }
                }
                changes.append(change)

            # Handle closed clients (separate from opened - not elif!)
            if closed_clients:
                change = {
                    "rule_id": rule_id,
                    "change_type": "clients_closed",
                    "clients_affected": {
                        "closed": closed_clients,
                        "count": len(closed_clients)
                    },
                    "rule_scope": {
                        "hotels": {
                            "type": curr_rule.get("lvl", {}).get("hot", {}).get("t", 0),
                            "list": curr_rule.get("lvl", {}).get("hot", {}).get("l", []) or []
                        },
                        "providers": {
                            "type": curr_rule.get("lvl", {}).get("prv", {}).get("t", 0),
                            "list": curr_rule.get("lvl", {}).get("prv", {}).get("l", []) or []
                        }
                    }
                }
                changes.append(change)

    return changes

# Run the clean analysis
clean_changes = analyze_rule_changes_clean(previous_week, current_week)

# Show ALL individual changes in clean format (not just first 3)
print("=== Individual Rule Changes ===")
for change in clean_changes:
    print(json.dumps(change, indent=2))
    print()

# Create summary report in clean format
openings_only = [c for c in clean_changes if c["change_type"] == "clients_opened"]
closings_only = [c for c in clean_changes if c["change_type"] == "clients_closed"]

# Calculate total products opened (client + hotel/provider combinations)
total_products_opened = 0
for change in openings_only:
    clients_count = change["clients_affected"]["count"]
    hotel_scope = len(change["rule_scope"]["hotels"]["list"]) if change["rule_scope"]["hotels"]["type"] == 1 else 1
    provider_scope = len(change["rule_scope"]["providers"]["list"]) if change["rule_scope"]["providers"]["type"] == 1 else 1
    total_products_opened += clients_count * hotel_scope * provider_scope

summary_report = {
    "analysis_period": "2025-09-22 to 2025-09-29",
    "total_rules_changed": len(clean_changes),
    "changes_by_type": {
        "clients_opened": len(openings_only),
        "clients_closed": len(closings_only)
    },
    "impact_summary": {
        "total_clients_opened": sum(c["clients_affected"]["count"] for c in openings_only),
        "total_clients_closed": sum(c["clients_affected"]["count"] for c in closings_only),
        "total_products_opened": total_products_opened,
        "rules_with_major_openings": len([c for c in openings_only if c["clients_affected"]["count"] >= 3])
    }
}

print("=== Summary Report ===")
print(json.dumps(summary_report, indent=2))

=== Individual Rule Changes ===
{
  "rule_id": "1008402",
  "change_type": "clients_closed",
  "clients_affected": {
    "closed": [
      "11945",
      "33533",
      "33238",
      "33534",
      "5226"
    ],
    "count": 5
  },
  "rule_scope": {
    "hotels": {
      "type": 1,
      "list": [
        "1812674",
        "221397",
        "1859825",
        "3505064",
        "3814800",
        "932913",
        "1838957",
        "221311"
      ]
    },
    "providers": {
      "type": 1,
      "list": [
        "XEN",
        "BLW"
      ]
    }
  }
}

{
  "rule_id": "464445",
  "change_type": "clients_opened",
  "clients_affected": {
    "opened": [
      "5226",
      "11945",
      "8550",
      "33238"
    ],
    "count": 4
  },
  "rule_scope": {
    "hotels": {
      "type": 2,
      "list": [
        "1167"
      ]
    },
    "providers": {
      "type": 1,
      "list": [
        "DNG",
        "BCONG"
      ]
    }
  }
}

{
  "rule_id": "464445",
  "change_type": "clients

In [21]:
# Step 1: Rule Comparison Setup & Function Definition
def analyze_rule_changes_clean(previous_rules_dict, current_rules_dict):
    """Simple analysis with clean output format - handles mixed opened+closed in single entry"""

    changes = []

    # Step 1: Rule Comparison Setup
    for rule_id in current_rules_dict:
        # Step 2: Rule Change Detection
        if rule_id not in previous_rules_dict:
            continue  # Skip new rules for now

        prev_rule = previous_rules_dict[rule_id]
        curr_rule = current_rules_dict[rule_id]

        # Extract client lists from both versions (lvl.cli.l)
        prev_client_list = prev_rule.get("lvl", {}).get("cli", {}).get("l", [])
        curr_client_list = curr_rule.get("lvl", {}).get("cli", {}).get("l", [])

        # Step 3: Client Set Analysis
        prev_clients = set(prev_client_list or [])
        curr_clients = set(curr_client_list or [])

        if prev_clients != curr_clients:
            # Calculate opened clients = clients removed from restriction (prev - curr)
            opened_clients = list(prev_clients - curr_clients)
            # Calculate closed clients = clients added to restriction (curr - prev)
            closed_clients = list(curr_clients - prev_clients)

            # Step 4: Change Classification
            if opened_clients and closed_clients:
                change_type = "clients_mixed"
            elif opened_clients:
                change_type = "clients_opened"
            else:
                change_type = "clients_closed"

            # Build clients_affected data
            clients_affected = {}
            if opened_clients:
                clients_affected["opened"] = opened_clients
                clients_affected["opened_count"] = len(opened_clients)
            if closed_clients:
                clients_affected["closed"] = closed_clients
                clients_affected["closed_count"] = len(closed_clients)

            # Step 5: Rule Scope Extraction
            rule_scope = {
                "hotels": {
                    "type": curr_rule.get("lvl", {}).get("hot", {}).get("t", 0),
                    "list": curr_rule.get("lvl", {}).get("hot", {}).get("l", []) or [],
                },
                "providers": {
                    "type": curr_rule.get("lvl", {}).get("prv", {}).get("t", 0),
                    "list": curr_rule.get("lvl", {}).get("prv", {}).get("l", []) or [],
                },
            }

            # Step 6: Result Formatting
            change = {
                "rule_id": rule_id,
                "change_type": change_type,
                "clients_affected": clients_affected,
                "rule_scope": rule_scope,
            }
            changes.append(change)

    return changes

In [22]:
# Step 2: Run Analysis
changes = analyze_rule_changes_clean(previous_week, current_week)
print(f"Found {len(changes)} rule changes")

Found 5 rule changes


In [23]:
# Step 3: Show Sample Results
print("=== Sample Rule Changes ===")
for change in changes[:3]:
    rule_id = change["rule_id"]
    change_type = change["change_type"]

    if change_type == "clients_mixed":
        opened = change["clients_affected"].get("opened_count", 0)
        closed = change["clients_affected"].get("closed_count", 0)
        print(f"Rule {rule_id}: MIXED - {opened} opened, {closed} closed")
    elif change_type == "clients_opened":
        opened = change["clients_affected"]["opened_count"]
        print(f"Rule {rule_id}: OPENED - {opened} clients")
    else:
        closed = change["clients_affected"]["closed_count"]
        print(f"Rule {rule_id}: CLOSED - {closed} clients")

=== Sample Rule Changes ===
Rule 1008402: CLOSED - 5 clients
Rule 464445: MIXED - 4 opened, 1 closed
Rule 395522: MIXED - 3 opened, 1 closed


In [24]:
# Step 4: Categorize Changes
mixed_rules = [c for c in changes if c["change_type"] == "clients_mixed"]
opened_only = [c for c in changes if c["change_type"] == "clients_opened"]
closed_only = [c for c in changes if c["change_type"] == "clients_closed"]

print("=== Change Categories ===")
print(f"Mixed changes (both opened + closed): {len(mixed_rules)} rules")
print(f"Opened only: {len(opened_only)} rules")
print(f"Closed only: {len(closed_only)} rules")

=== Change Categories ===
Mixed changes (both opened + closed): 3 rules
Opened only: 0 rules
Closed only: 2 rules


In [25]:
# Step 5: Summary Statistics
total_clients_opened = sum(
    c["clients_affected"].get("opened_count", 0) for c in changes
)
total_clients_closed = sum(
    c["clients_affected"].get("closed_count", 0) for c in changes
)

summary = {
    "analysis_period": "2025-09-22 to 2025-09-29",
    "total_rules_changed": len(changes),
    "changes_by_type": {
        "clients_mixed": len(mixed_rules),
        "clients_opened": len(opened_only),
        "clients_closed": len(closed_only)
    },
    "impact_summary": {
        "total_clients_opened": total_clients_opened,
        "total_clients_closed": total_clients_closed,
        "rules_with_major_openings": len([c for c in changes if c["clients_affected"].get("opened_count", 0) >= 3])
    }
}

print("=== Summary Report ===")
print(json.dumps(summary, indent=2))

=== Summary Report ===
{
  "analysis_period": "2025-09-22 to 2025-09-29",
  "total_rules_changed": 5,
  "changes_by_type": {
    "clients_mixed": 3,
    "clients_opened": 0,
    "clients_closed": 2
  },
  "impact_summary": {
    "total_clients_opened": 13,
    "total_clients_closed": 11,
    "rules_with_major_openings": 3
  }
}


In [27]:
print(json.dumps(changes, indent=2))

[
  {
    "rule_id": "1008402",
    "change_type": "clients_closed",
    "clients_affected": {
      "closed": [
        "11945",
        "33533",
        "33238",
        "33534",
        "5226"
      ],
      "closed_count": 5
    },
    "rule_scope": {
      "hotels": {
        "type": 1,
        "list": [
          "1812674",
          "221397",
          "1859825",
          "3505064",
          "3814800",
          "932913",
          "1838957",
          "221311"
        ]
      },
      "providers": {
        "type": 1,
        "list": [
          "XEN",
          "BLW"
        ]
      }
    }
  },
  {
    "rule_id": "464445",
    "change_type": "clients_mixed",
    "clients_affected": {
      "opened": [
        "5226",
        "11945",
        "8550",
        "33238"
      ],
      "opened_count": 4,
      "closed": [
        "11"
      ],
      "closed_count": 1
    },
    "rule_scope": {
      "hotels": {
        "type": 2,
        "list": [
          "1167"
        ]
     

In [28]:
# Step 1: Prepare Volume Analysis Setup
print("=== Volume Analysis Setup ===")

# Focus on rules with opened clients (where restrictions were lifted)
rules_with_openings = [c for c in changes if c["clients_affected"].get("opened_count", 0) > 0]

print(f"Found {len(rules_with_openings)} rules with client openings to analyze")

# Prepare volume analysis data for each rule
volume_analysis_prep = []

for rule_change in rules_with_openings:
    rule_id = rule_change["rule_id"]
    opened_clients = rule_change["clients_affected"]["opened"]
    hotel_scope = rule_change["rule_scope"]["hotels"]
    provider_scope = rule_change["rule_scope"]["providers"]

    # Prepare the analysis context
    analysis_context = {
        "rule_id": rule_id,
        "opened_clients": opened_clients,
        "opened_count": len(opened_clients),
        "hotel_constraint": {
            "type": hotel_scope["type"],
            "list": hotel_scope["list"],
            "description": f"Hotel type {hotel_scope['type']}: {len(hotel_scope['list'])} items" if hotel_scope["list"] else f"Hotel type {hotel_scope['type']}: All hotels"
        },
        "provider_constraint": {
            "type": provider_scope["type"],
            "list": provider_scope["list"],
            "description": f"Provider type {provider_scope['type']}: {len(provider_scope['list'])} items" if provider_scope["list"] else f"Provider type {provider_scope['type']}: All providers"
        }
    }

    volume_analysis_prep.append(analysis_context)

# Show summary of what we'll analyze
print("\nRules ready for volume analysis:")
for prep in volume_analysis_prep:
    print(f"Rule {prep['rule_id']}: {prep['opened_count']} clients opened")
    print(f"  - {prep['hotel_constraint']['description']}")
    print(f"  - {prep['provider_constraint']['description']}")
    print(f"  - Clients: {prep['opened_clients'][:3]}{'...' if len(prep['opened_clients']) > 3 else ''}")
    print()

=== Volume Analysis Setup ===
Found 3 rules with client openings to analyze

Rules ready for volume analysis:
Rule 464445: 4 clients opened
  - Hotel type 2: 1 items
  - Provider type 1: 2 items
  - Clients: ['5226', '11945', '8550']...

Rule 395522: 3 clients opened
  - Hotel type 2: 1 items
  - Provider type 1: 2 items
  - Clients: ['11732', '11731', '3713']

Rule 395308: 6 clients opened
  - Hotel type 2: 1 items
  - Provider type 1: 2 items
  - Clients: ['5060', '11656', '11732']...



In [29]:
# Step 2: Build SQL Query Generation
def build_rule_volume_query(rule_context):
    """Build SQL query for volume analysis based on rule constraints"""

    rule_id = rule_context["rule_id"]
    opened_clients = rule_context["opened_clients"]
    hotel_constraint = rule_context["hotel_constraint"]
    provider_constraint = rule_context["provider_constraint"]

    # Build WHERE conditions based on rule scope
    conditions = []

    # Client filter - focus on opened clients for attribution
    client_list = "', '".join(opened_clients)
    attribution_filter = f"b.credential_id IN ('{client_list}')"

    # Hotel constraint
    if hotel_constraint["type"] == 1:  # Specific hotels
        hotel_list = "', '".join(hotel_constraint["list"])
        hotel_filter = f"b.hotel_id IN ('{hotel_list}')"
    elif hotel_constraint["type"] == 2:  # Hotel chains
        chain_list = "', '".join(hotel_constraint["list"])
        hotel_filter = f"hh.chain_id IN ('{chain_list}')"
    else:  # All hotels (type 0)
        hotel_filter = "1=1"

    # Provider constraint
    if provider_constraint["type"] == 1:  # Specific providers
        provider_list = "', '".join(provider_constraint["list"])
        provider_filter = f"b.provider_id IN ('{provider_list}')"
    elif provider_constraint["type"] == 2:  # Indirect providers
        provider_filter = "NOT hp.direct"
    elif provider_constraint["type"] == 3:  # Direct providers
        provider_filter = "hp.direct"
    else:  # All providers (type 0)
        provider_filter = "1=1"

    # Build the complete query adapted from your existing framework
    query = f"""
    WITH
        rule_scope AS (
            SELECT '{rule_id}' as rule_id
        ),
        weekly_sales AS (
            SELECT
                r.rule_id,
                c.start_of_week,
                c.end_of_week,
                -- Attribution analysis: revenue from opened clients only
                COALESCE(SUM(CASE WHEN {attribution_filter} THEN b.net_agency ELSE 0 END), 0) as attribution_revenue,
                -- Total revenue for this product scope (all clients)
                COALESCE(SUM(b.net_agency), 0) as total_revenue,
                -- Count of bookings for context
                COUNT(CASE WHEN {attribution_filter} THEN 1 END) as attribution_bookings,
                COUNT(*) as total_bookings
            FROM
                reports_calendar c
                JOIN reports_bookingsfinancialdata b ON b.booking_date::date = c.date
                LEFT JOIN hotel_hotel hh ON hh.id = b.hotel_id
                LEFT JOIN hotel_provider hp ON hp.code = b.provider_id
                LEFT JOIN clients_credential cre ON cre.id = b.credential_id
                LEFT JOIN clients_client cli ON cli.id = cre.client_id
                CROSS JOIN rule_scope r
            WHERE
                cli.region_id <> 'VECI'
                AND cli.id <> 83
                AND c.week_offset IN (-1, 0)  -- Previous week and current week
                AND {hotel_filter}
                AND {provider_filter}
            GROUP BY
                r.rule_id,
                c.start_of_week,
                c.end_of_week
        ),
        weekly_comparison AS (
            SELECT
                rule_id,
                start_of_week,
                end_of_week,
                attribution_revenue,
                total_revenue,
                attribution_bookings,
                total_bookings,
                LAG(attribution_revenue) OVER (PARTITION BY rule_id ORDER BY start_of_week) as prev_attribution_revenue,
                LAG(total_revenue) OVER (PARTITION BY rule_id ORDER BY start_of_week) as prev_total_revenue
            FROM weekly_sales
        )
    SELECT
        rule_id,
        start_of_week,
        end_of_week,
        attribution_revenue,
        total_revenue,
        attribution_bookings,
        total_bookings,
        prev_attribution_revenue,
        prev_total_revenue,
        -- Attribution rate: what % of total revenue came from opened clients
        CASE WHEN total_revenue > 0 
             THEN (attribution_revenue / total_revenue::numeric) 
             ELSE 0 END as attribution_rate,
        -- Growth calculation
        CASE WHEN prev_total_revenue = 0 AND total_revenue > 0 THEN 1.0
             WHEN prev_total_revenue = 0 AND total_revenue = 0 THEN NULL
             ELSE (total_revenue - prev_total_revenue) / ABS(prev_total_revenue)::numeric
        END as growth_rate
    FROM weekly_comparison
    WHERE prev_attribution_revenue IS NOT NULL  -- Only show current week with comparison
    ORDER BY rule_id, start_of_week;
    """

    return query

# Generate queries for all rules with openings
print("=== Generated SQL Queries ===")
for i, rule_context in enumerate(volume_analysis_prep):
    query = build_rule_volume_query(rule_context)
    print(f"Query for Rule {rule_context['rule_id']}:")
    print(f"-- {rule_context['opened_count']} clients opened: {rule_context['opened_clients']}")
    print(query[:500] + "..." if len(query) > 500 else query)
    print("\\n" + "="*50 + "\\n")

    if i >= 1:  # Show first 2 queries as examples
        print(f"... (and {len(volume_analysis_prep) - 2} more queries)")
        break

=== Generated SQL Queries ===
Query for Rule 464445:
-- 4 clients opened: ['5226', '11945', '8550', '33238']

    WITH
        rule_scope AS (
            SELECT '464445' as rule_id
        ),
        weekly_sales AS (
            SELECT
                r.rule_id,
                c.start_of_week,
                c.end_of_week,
                -- Attribution analysis: revenue from opened clients only
                COALESCE(SUM(CASE WHEN b.credential_id IN ('5226', '11945', '8550', '33238') THEN b.net_agency ELSE 0 END), 0) as attribution_revenue,
                -- Total revenue for this product scope...
\n==================================================\n
Query for Rule 395522:
-- 3 clients opened: ['11732', '11731', '3713']

    WITH
        rule_scope AS (
            SELECT '395522' as rule_id
        ),
        weekly_sales AS (
            SELECT
                r.rule_id,
                c.start_of_week,
                c.end_of_week,
                -- Attribution analysis: 